# Spaceship Titanic — Exploratory Data Analysis

**Notebook 01 of 3** — EDA and initial data understanding.

## Competition Overview

The Spaceship Titanic was hit by a spacetime anomaly en route to three habitable exoplanets. Almost half of the passengers were transported to an alternate dimension. Our task is to predict which passengers were transported, based on records recovered from the ship's damaged computer system.

This is a **binary classification** problem with the target variable `Transported` (True/False).

## Dataset

- **Training set:** ~8,700 passengers with known `Transported` labels
- **Test set:** ~4,300 passengers with unknown labels (for Kaggle submission)

## Notebook Structure

1. Setup and Data Loading
2. Initial Data Inspection
3. Target Variable Analysis
4. Missing Values Analysis
5. Categorical Features
6. Numerical Features
7. Feature Relationships with Target
8. Key Findings and Next Steps

## 1. Setup and Data Loading

**Goal:** Import libraries and load both `train.csv` and `test.csv`.

**Hints:**
- Standard imports you'll need: `numpy`, `pandas`, `matplotlib.pyplot`, `seaborn`
- Useful display settings: `pd.set_option('display.max_columns', None)` so wide dataframes show all columns
- Consider setting a `RANDOM_STATE = 42` constant for reproducibility later
- Load **both** `train` and `test` from `../data/raw/`. Even though we won't compute stats on test, we want to inspect it to confirm columns match and plan preprocessing
- After loading, print `.shape` for both and verify which column exists in train but not test (that's your target)

## 2. Initial Data Inspection

**Goal:** Get a first look at the data — column types, sample rows, basic statistics.

**Hints:**
- `.head()` to see the first few rows
- `.info()` to see dtypes and non-null counts
- `.describe()` for numerical summary stats
- `.describe(include='object')` for categorical columns

**Things to look for:**
- What are the column types? (numerical vs categorical vs boolean)
- Which columns have missing values?
- Do any numerical columns have suspicious min/max values (e.g., negative spending)?
- Does `PassengerId` have a pattern? (look closely at the format)

### Observations

*Fill in what you notice about the data here.*

## 3. Target Variable Analysis

**Goal:** Understand the distribution of the target variable `Transported`.

**Why this matters:** If the classes are very imbalanced (e.g., 95%/5%), accuracy becomes a misleading metric and we'd need to consider stratified sampling, class weights, or oversampling. If they're balanced (~50/50), accuracy is a reasonable default metric.

**Hints:**
- `.value_counts()` for raw counts
- `.value_counts(normalize=True)` for proportions
- A simple bar plot makes the balance visually obvious (`sns.countplot` or `.value_counts().plot(kind='bar')`)

### Observations

*Is the target balanced or imbalanced? What does that imply for our metric choice?*

## 4. Missing Values Analysis

**Goal:** Identify how much is missing per column and whether missingness itself might be informative.

**Why this matters:** Before we can impute, we need to know the scope. Also, sometimes a missing value is *itself* a signal (e.g., if wealthy passengers are less likely to report their cabin, missingness correlates with the target).

**Hints:**
- `.isnull().sum()` gives missing counts per column
- Divide by `len(train)` and multiply by 100 for percentages
- Consider building a small summary DataFrame with count and percentage columns, sorted descending
- A heatmap of `df.isnull()` (using `sns.heatmap`) can reveal patterns — do rows tend to have multiple missing values together, or are missing values scattered?

**Stretch exercise:** For each column with missing values, check whether `Transported` rate differs between rows where that column is missing vs not missing. If it does, the missingness itself carries predictive signal.

### Observations

*How much is missing per column? Any patterns in the missingness?*

## 5. Categorical Features

**Goal:** Understand each categorical column's distribution and any obvious structure.

**Categorical columns:** `HomePlanet`, `CryoSleep`, `Cabin`, `Destination`, `VIP`, `Name`, and also `PassengerId` (which has internal structure).

**For each, we want to know:**
- What are the unique values?
- Is the distribution balanced or dominated by one category?
- Is there internal structure to parse out? (e.g., `Cabin` is `Deck/Num/Side`, `PassengerId` is `Group_NumberInGroup`)

### 5.1 PassengerId

**Hint:** The format is `gggg_pp` where `gggg` is the group ID and `pp` is the position within the group. People in the same group are often family or travel companions. This is a goldmine for feature engineering later (group size, traveling alone vs. with family, etc.) but for now, just confirm the format and count distinct groups.

### 5.2 HomePlanet

**Hint:** `.value_counts()` will show you the three planets and their counts. Consider a simple bar plot.

### 5.3 CryoSleep

**Hint:** CryoSleep is a boolean. Passengers in cryosleep are confined to their cabins — so we should expect their spending columns to all be zero. Worth checking this assumption.

### 5.4 Cabin

**Hint:** Format is `Deck/Number/Side` (e.g., `B/0/P`). You already parsed this in your previous notebook — use `.str.split('/')` to split into three new columns: `Cabin_Deck`, `Cabin_Num`, `Cabin_Side`. Then `.value_counts()` on each.

**Watch out for:** missing values. `.str.split` returns NaN for NaN input, which is what we want, but be careful if you convert `Cabin_Num` to int (NaN doesn't convert to int cleanly).

### 5.5 Destination

**Hint:** Three destinations. Simple `.value_counts()` and a bar plot.

### 5.6 VIP

**Hint:** Boolean. Expect this to be very imbalanced (few VIPs). Worth noting how rare VIP is — with so few examples, this feature may not be very predictive on its own.

### 5.7 Name

**Hint:** Names are mostly unique. The interesting signal might be in **last names** (family groupings). Try splitting into `FirstName` and `LastName` using `.str.split(' ')` and see how many passengers share a last name. Later, we may engineer a family-size feature from this combined with the group ID.

### Observations

*Summarize what you learned about the categorical features.*

## 6. Numerical Features

**Goal:** Understand each numerical column's distribution.

**Numerical columns:** `Age`, `RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`.

**Key questions:**
- What's the shape of the distribution? (normal, skewed, bimodal?)
- Are there outliers?
- Do the spending columns share a common pattern? (spoiler: lots of zeros, long tails)

### 6.1 Age

**Hint:** Histogram (`sns.histplot` or `plt.hist`). Also consider a boxplot. What's the age range? Are there children (age 0) that might behave differently from adults?

### 6.2 Spending Columns (RoomService, FoodCourt, ShoppingMall, Spa, VRDeck)

**Hint:** These columns will be heavily right-skewed (most passengers spent $0, a few spent thousands). Consider:
- A histogram for each — they'll look ugly because of the zeros
- Try a log-scale y-axis, or plot `np.log1p(column)` (log of 1+x, which handles zeros)
- The proportion of zeros per column is itself an interesting statistic

**Feature engineering idea for later:** a `TotalSpend` column (sum of all five), and a `HasSpent` boolean.

### Observations

*What do the numerical distributions look like? Any transformations we should plan for?*

## 7. Feature Relationships with Target

**Goal:** For each feature, does it seem related to `Transported`? This is where we find the signals our model will exploit.

**Important distinction to keep in mind:** there's a difference between **count** (how many passengers with this value were transported) and **rate / proportion** (what fraction of passengers with this value were transported). The rate is usually what matters for modeling. For example, 'Deck G has the most transported passengers' is usually because Deck G has the most passengers — not because G has a high transport rate.

### 7.1 Categorical features vs Transported

**Hint:** For each categorical feature (`HomePlanet`, `CryoSleep`, `Destination`, `VIP`, `Cabin_Deck`, `Cabin_Side`), compute the transport **rate** per category.

Two useful patterns:
- `df.groupby('HomePlanet')['Transported'].mean()` — since `Transported` is boolean, mean gives you the rate of True
- `sns.countplot(x='HomePlanet', hue='Transported', data=train)` — stacked bars showing the split

**Look for:** features where the transport rate differs noticeably across categories. Those are your predictive features.

### 7.2 Numerical features vs Transported

**Hint:** For each numerical feature:
- Boxplot split by target: `sns.boxplot(x='Transported', y='Age', data=train)`
- Or overlapping histograms
- For spending columns, compare median spending (not mean — mean is distorted by outliers) between Transported=True and False

**Look for:** features where the distribution shifts between the two classes. If transported passengers spend less on RoomService on average, that's a signal.

### 7.3 Correlation heatmap

**Hint:** For the numerical columns, `train.corr()` gives a correlation matrix, and `sns.heatmap` visualizes it. Include the target (convert `Transported` to int first: `train['Transported'].astype(int)`).

**Note:** Correlation only captures *linear* relationships. A feature can be strongly predictive without being linearly correlated with the target.

### Observations

*Which features look most predictive? Any surprises?*

## 8. Key Findings and Next Steps

### Summary of Findings

*Fill in after completing EDA. What are the 3-5 most important things you learned?*

### Feature Engineering Ideas

Based on what we saw, we'll engineer the following features in notebook `02_feature_engineering.ipynb`:

- *(fill in as you complete EDA — e.g., group size from PassengerId, total spend, cabin parsing, family size from surname, etc.)*

### Preprocessing Plan

- Missing value imputation strategy: *(fill in based on what you saw)*
- Encoding strategy for categoricals: *(one-hot? target encoding?)*
- Any transformations for skewed numerical features: *(log1p for spending?)*